In [1]:
import argparse
import logging
import os
import random
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch import optim
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

import wandb
from evaluate import evaluate
from unet.unet_model import UNet
from utils.data_loading import *
from utils.dice_score import dice_loss

/home/lqmeyers/anaconda3/envs/DLC/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
classes = 4 #put into yaml 
bilinear = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Using device {device}')

# Change here to adapt to your data
# n_channels=3 for RGB images
# n_classes is the number of probabilities you want to get per pixel
model = UNet(n_channels=3, n_classes=classes, bilinear=bilinear)
model = model.to(memory_format=torch.channels_last)

logging.info(f'Network:\n'
                f'\t{model.n_channels} input channels\n'
                f'\t{model.n_classes} output channels (classes)\n'
                f'\t{"Bilinear" if model.bilinear else "Transposed conv"} upscaling')

model.to(device=device)

# -e 30 -b 10 -c 2 -v 15 -s 1 -l 1e-4


dir_checkpoint = Path('./checkpoints/')
model = model
device = device 
epochs = 5
batch_size = 1
learning_rate = 1e-4
val_percent = 0.15
save_checkpoint = True
img_scale = 1
amp = False
weight_decay = 1e-8
momentum = 0.999
gradient_clipping = 1.0

In [3]:

class LabelMeDataset(Dataset):
    def __init__(self, images_dir: str, mask_xml_dir: str, mask_dir:str, scale: float = 1.0, mask_suffix: str = ''):
        self.images_dir = images_dir
        self.mask_dir = mask_dir
        self.mask_xml_dir = mask_xml_dir
        assert 0 < scale <= 1, 'Scale must be between 0 and 1'
        self.scale = scale
        self.mask_suffix = mask_suffix

        self.ids = [splitext(file)[0] for file in listdir(images_dir) if isfile(join(images_dir, file)) and not file.startswith('.')]
        
        if not self.ids:
            raise RuntimeError(f'No input file found in {images_dir}, make sure you put your images there')

        logging.info(f'Creating dataset with {len(self.ids)} examples')
        logging.info('Scanning mask files to determine unique values')
        # with Pool() as p:
        #     unique = list(tqdm(
        #         p.imap(partial(unique_mask_values, mask_dir=self.mask_dir, mask_suffix=self.mask_suffix), self.ids),
        #         total=len(self.ids)
        #     ))

        # self.mask_values = list(sorted(np.unique(np.concatenate(unique), axis=0).tolist()))
        # logging.info(f'Unique mask values: {self.mask_values}')

    def __len__(self):
        return len(self.ids)

    @staticmethod
    def preprocess( pil_img, scale, is_mask,mask_values=8):
        w, h = pil_img.size
        newW, newH = int(scale * w), int(scale * h)
        assert newW > 0 and newH > 0, 'Scale is too small, resized images would have no pixel'
        pil_img = pil_img.resize((newW, newH), resample=Image.NEAREST if is_mask else Image.BICUBIC)
        img = np.asarray(pil_img)

        if is_mask: #generate class array? 
            mask = np.zeros((newH, newW), dtype=np.int64)
            for i, v in enumerate(mask_values):
                if img.ndim == 2:
                    mask[img == v] = i
                else:
                    mask[(img == v).all(-1)] = i

            return mask

        else:
            if img.ndim == 2:
                img = img[np.newaxis, ...]
            else:
                img = img.transpose((2, 0, 1))

            if (img > 1).any():
                img = img / 255.0

            return img
    
    def assemble_mask_from_xml(self,xml_file,mask_path):
        '''assembles a multiclass numpy array by reading multiple binary masks
        stored in the mask paths folder using filnames contained in the xml file'''
        xml_dict = get_xml_mask_dict(xml_file)
        mask_list = []
        i = 0 
        for key in xml_dict:
            path = os.path.join(mask_path+xml_dict[key])
            img = Image.open(path).convert('L')
            normalized_img = (img - np.min(img)) / (np.max(img) - np.min(img))
            # Convert the normalized image to a binary mask
            array = (normalized_img > 0.5).astype(np.uint8)
           # array =  np.array(array).transpose((1, 0,))
            mask_list.append(array)
            if i == 0:
                background_array = np.zeros_like(array)
                background_array = np.bitwise_or(background_array,array)
            else:
                background_array = np.bitwise_or(background_array,array)
            i+= 1 
        #mask_list.append(background_array)
        full_array = np.array(mask_list)
        return full_array


    def __getitem__(self, idx):
        name = self.ids[idx]
        path = os.path.join(self.mask_xml_dir,name + '.*')
        xml_file =  glob(path)
        img_file = glob(os.path.join(self.images_dir,name + '.*'))

        #mask_files = list(self.mask_dir.glob(name + self.mask_suffix + '.*'))
        assert len(img_file) == 1, f'Either no image or multiple images found for the ID {name}: {img_file}'
        assert len(xml_file) == 1, f'Either no mask or multiple masks found for the ID {name}: {xml_file}'
        
        #build array from multiple masks as specified in xml 
        mask_array = self.assemble_mask_from_xml(xml_file[0],self.mask_dir)      

        #mask = load_image(mask_file[0])
        img = load_image(img_file[0])
        img = self.preprocess(img, self.scale, is_mask=False)
     
        assert img[0].shape == mask_array[0].shape, \
            f'Image and mask {name} should be the same size, but are {img[0].shape} and {mask_array[0].shape}'

        #mask = self.preprocess(self.mask_values, mask, self.scale, is_mask=True)

        return {
            'image': torch.as_tensor(img.copy()).float().contiguous(),
            'mask': torch.as_tensor(mask_array.copy()).float().contiguous()
        }


In [4]:
# 1. Create dataset
# dir_img = Path('/home/lqmeyers/paintDetect/data/images/training/')
# dir_mask = Path('/home/lqmeyers/paintDetect/data/full_masks/training/')
# dataset = BasicDataset(dir_img, dir_mask, img_scale)

dir_img = '/home/lqmeyers/SLEAP_files/Bee_imgs/baby_bee_imgs/CVAT_sample/'
dir_mask = '/home/lqmeyers/CVAT/babyBees3perID/Label_Me_3.0/default/Masks/'
dir_xml = '/home/lqmeyers/CVAT/babyBees3perID/Label_Me_3.0/default/'
dataset = LabelMeDataset(dir_img,dir_xml,dir_mask,img_scale)


# 2. Split into train / validation partitions
n_val = int(len(dataset) * val_percent)
n_train = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(0))
print(train_set.__getitem__(10))
image, mask = train_set.__getitem__(10).values()
for o in train_set.__getitem__(10).values():
    print(o.shape)

# 3. Create data loaders
loader_args = dict(batch_size=batch_size, num_workers=os.cpu_count(), pin_memory=True)
train_loader = DataLoader(train_set, shuffle=True, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, drop_last=True, **loader_args)
print("number of batches of validation rounds is "+str(len(val_loader)))



{'image': tensor([[[0.8000, 0.8078, 0.8157,  ..., 0.8392, 0.8510, 0.8510],
         [0.8118, 0.8157, 0.8196,  ..., 0.8392, 0.8471, 0.8510],
         [0.8196, 0.8235, 0.8235,  ..., 0.8471, 0.8510, 0.8510],
         ...,
         [0.7490, 0.7451, 0.7490,  ..., 1.0000, 1.0000, 1.0000],
         [0.7333, 0.7216, 0.7255,  ..., 1.0000, 1.0000, 1.0000],
         [0.7333, 0.7216, 0.7294,  ..., 1.0000, 1.0000, 1.0000]],

        [[0.9765, 0.9843, 0.9882,  ..., 0.9922, 0.9922, 0.9922],
         [0.9882, 0.9922, 0.9922,  ..., 0.9922, 0.9882, 0.9922],
         [0.9922, 0.9961, 0.9961,  ..., 0.9882, 0.9922, 0.9922],
         ...,
         [0.9255, 0.9216, 0.9137,  ..., 1.0000, 1.0000, 1.0000],
         [0.9098, 0.8980, 0.8902,  ..., 1.0000, 1.0000, 1.0000],
         [0.9098, 0.8980, 0.8863,  ..., 1.0000, 1.0000, 1.0000]],

        [[0.9882, 0.9961, 0.9922,  ..., 0.9725, 0.9765, 0.9765],
         [1.0000, 1.0000, 0.9961,  ..., 0.9725, 0.9725, 0.9765],
         [0.9961, 1.0000, 1.0000,  ..., 0.9725, 

In [5]:

# 4. Set up the optimizer, the loss, the learning rate scheduler and the loss scaling for AMP
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

#optimizer = optim.RMSprop(model.parameters(),
                        #    lr=learning_rate, weight_decay=weight_decay, momentum=momentum, foreach=True)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5,factor=.5,min_lr=1e-6)  # goal: maximize Dice score
grad_scaler = torch.cuda.amp.GradScaler(enabled=amp)
criterion = nn.CrossEntropyLoss()
global_step = 0

image = image.unsqueeze(0)
mask = mask.unsqueeze(0)
image = image.to(device=device, dtype=torch.float32)
mask_true = mask.to(device=device, dtype=torch.float)

# predict the mask
model.eval()
mask_pred = model(image)
print(criterion(mask_pred,mask_true))

tensor(0.3315, device='cuda:0', grad_fn=<DivBackward1>)


In [6]:
def evaluate(net, dataloader, device, amp):
    net.eval()
    num_val_batches = len(dataloader)
    dice_score = 0
    val_loss = 0

    # iterate over the validation set
    with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
        for batch in tqdm(dataloader, total=num_val_batches, desc='Validation round', unit='batch', leave=False):
            image, mask_true = batch['image'], batch['mask']

            # move images and labels to correct device and type
            image = image.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
            mask_true = mask_true.to(device=device, dtype=torch.float)

            # predict the mask
            mask_pred = net(image)
            
            # if net.n_classes == 1:
            #     assert mask_true.min() >= 0 and mask_true.max() <= 1, 'True mask indices should be in [0, 1]'
            #     mask_pred = (F.sigmoid(mask_pred) > 0.5).float()
            #     # compute the Dice score
            #     dice_score += dice_coeff(mask_pred, mask_true, reduce_batch_first=False)
            # else:
            assert mask_true.min() >= 0 and mask_true.max() < net.n_classes, 'True mask indices should be in [0, n_classes['
            # convert to one-hot format
            #mask_true = F.one_hot(mask_true, net.n_classes).permute(0, 3, 1, 2).float()  #WE DONT WANT ONE HOT BECAUSE IT MAKES CLASSES MUTUALLY EXCLUSIVE 
            #mask_pred = F.one_hot(mask_pred.argmax(dim=1), net.n_classes).permute(0, 3, 1, 2).float()
            # compute the Dice score, ignoring background
            #dice_score += multiclass_dice_coeff(mask_pred[:, 1:], mask_true[:, 1:], reduce_batch_first=False)
            val_loss += criterion(mask_pred,mask_true)
            dice_score = val_loss

    net.train()
    return dice_score / max(num_val_batches, 1)


In [6]:

# 5. Begin training
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0
    with tqdm(total=n_train, desc=f'Epoch {epoch}/{epochs}', unit='img') as pbar:
        for batch in train_loader:
            images, true_masks = batch['image'], batch['mask']

            assert images.shape[1] == model.n_channels, \
                f'Network has been defined with {model.n_channels} input channels, ' \
                f'but loaded images have {images.shape[1]} channels. Please check that ' \
                'the images are loaded correctly.'

            images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
            true_masks = true_masks.to(device=device, dtype=torch.float32)

            with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
                masks_pred = model(images)
                #if model.n_classes == 1:
                    #loss = criterion(masks_pred.squeeze(1), true_masks.float())
                    #loss += dice_loss(F.sigmoid(masks_pred.squeeze(1)), true_masks.float(), multiclass=False)
                #@else:
                loss = criterion(masks_pred, true_masks)
                #print(loss.shape)
                #loss += dice_loss(
                    #F.softmax(masks_pred, dim=1).float(),
                    #F.one_hot(true_masks, model.n_classes).permute(0, 3, 1, 2).float(),
                    #multiclass=True
                    # )

            optimizer.zero_grad(set_to_none=True)
            grad_scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
            grad_scaler.step(optimizer)
            grad_scaler.update()

            pbar.update(images.shape[0])
            global_step += 1
            epoch_loss += loss.item()
          
            pbar.set_postfix(**{'loss (batch)': loss.item()})

            # Evaluation round
            division_step = (n_train // (5 * batch_size))
            if division_step > 0:
                if global_step % division_step == 0:
                    histograms = {}
                    for tag, value in model.named_parameters():
                        tag = tag.replace('/', '.')
                        if not (torch.isinf(value) | torch.isnan(value)).any():
                            histograms['Weights/' + tag] = wandb.Histogram(value.data.cpu())
                        if not (torch.isinf(value.grad) | torch.isnan(value.grad)).any():
                            histograms['Gradients/' + tag] = wandb.Histogram(value.grad.data.cpu())

                    val_score = evaluate(model, val_loader, device, amp)
                    #print("score of validation round is "+str(val_score))
                    scheduler.step(val_score)

                

Epoch 1/5:  79%|███████▊  | 128/163 [00:35<00:06,  5.21img/s, loss (batch)=0.276]